# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [mlcroissant](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described via a Croissant schema and available here:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL to the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Spatial coverage: {metadata.spatialCoverage}")
print(f"Temporal coverage: {metadata.temporalCoverage}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available RecordSets and inspect their structure
record_sets = dataset.record_sets
print(f"Number of RecordSets: {len(record_sets)}\n")
if not record_sets:
    print("No RecordSets found in the current Croissant schema metadata.")
else:
    for i, recset in enumerate(record_sets):
        print(f"[{i}] RecordSet: {recset['@id']}")
        print(f"    Name: {recset.get('name')}")
        print(f"    Description: {recset.get('description')}")
        print("    Fields:")
        for field in recset.get('field', []):
            print(f"       - {field['@id']} : {field.get('name')}")
        print()

For demonstration, we will attempt to view the records in the first available RecordSet (if any):

In [ ]:
# Pick a RecordSet to explore, if any exist
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"Records from RecordSet: {first_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(f"Record {i}: {record}")
        if i >= 2:
            print("... [output truncated after 3 records]")
            break
else:
    print('No record sets found to preview records from.')

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames for further analysis. All RecordSet and field references use their `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}.")

if dataframes:
    # Pick the first DataFrame for demonstration
    demo_record_set_id = next(iter(dataframes))
    print(f"\nSample records from RecordSet '{demo_record_set_id}':")
    display(dataframes[demo_record_set_id].head())
else:
    print("No dataframes created: dataset may not have accessible tabular data via Croissant recordSets.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from one of the extracted DataFrames and perform filtering, normalization, and grouping. All field and group references are via their `@id`.

*If no numeric fields are present, this cell demonstrates the structure for EDA with placeholder variables.*

In [ ]:
# Pick the first DataFrame for EDA (if any loaded above)
if dataframes:
    df = dataframes[demo_record_set_id]
    print(f"Exploring DataFrame from RecordSet: {demo_record_set_id}")
    # Try to detect a numeric field by checking dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a categorical/text/grouping field if available
        possible_group_fields = [col for col in df.columns if col != numeric_field_id]
        group_field_id = None
        for col in possible_group_fields:
            if df[col].dtype == object and df[col].nunique() < df.shape[0] // 2:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found; skipping EDA steps.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and group statistics, if present in the DataFrame.

In [ ]:
import matplotlib.pyplot as plt
_ = plt.style.use('seaborn-v0_8')

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # Boxplot by group if group field is found
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric or group field for visualization.")

## 6. Conclusion
This notebook illustrated loading and inspecting a Croissant FAIR² dataset package using the `mlcroissant` library, navigating metadata, and referencing all entities by their `@id`. For more complex, multi-table, or hierarchical Croissant datasets, adapt the steps to explore all available RecordSets and linked resources.

Explore the dataset further using field `@id` references for processing, model preparation, or advanced analytics.